# 7. Chuẩn hóa, trích xuất BibTeX và khử trùng 2 tầng

## 7.1 Chuẩn hóa LaTeX & Toán học (`normalize.py`)

- Gỡ các wrapper trình bày (`\textbf`, `\emph`, ...) giữ lại ngữ nghĩa bên trong; xóa các lệnh bố cục (`\centering`, `\toprule`, `\midrule`, `\bottomrule`, tùy chọn `[htpb]`...).
- Chuẩn hóa toàn bộ inline math (`$...$`, `$...$`) về `$...$`, và toàn bộ block math (`$$...$$`, `
$$
...
$$
`, `equation*`, `align`, `gather`, `multline`, ...) về `\begin{equation}...\end{equation}`.

## 7.2 Khử trùng 2 tầng xuyên version (`bibliography.py` & `deduplicate.py`)

- **Tầng 1 - Reference entries**: hợp nhất các bản ghi từ `.bib`, `\bibitem` và fallback `.bbl` theo định danh mạnh (`doi`, `eprint` arXiv) hoặc chữ ký `(title, authors)`, gộp (union) trường mới từ các version sau và **đổi tên mọi lệnh `\cite{}`** (kể cả trong môi trường toán) về khóa đại diện duy nhất.
- **Tầng 2 - Full-text elements**: chuẩn hóa từng nút trước khi băm SHA-256 theo `(arxiv_id, kind, normalized_content)` để dùng chung `element_id` ở bảng `elements`, giữ riêng từng cạnh `(version, element_id, parent_id)` ở bảng `hierarchy`.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
data = json.loads((Path.cwd().parent / 'report_summary.json').read_text(encoding='utf-8'))
scope, m1, m2, bench = data['scope'], data['m1'], data['m2'], data['m2_benchmark']
plt.rcParams.update({'figure.figsize': (9.2, 4.2), 'axes.spines.top': False, 'axes.spines.right': False, 'figure.dpi': 120})
print(f"Toàn nhóm (5 thành viên): {scope['papers']:,} paper / {scope['versions']:,} version | Cập nhật: {data['report_date']}")

In [ ]:
from parser.normalize import normalize
from parser.bibliography import parse_bibitems, unify_entries
from parser.hierarchy import parse_hierarchy
from parser.deduplicate import deduplicate

v1_tex = r"""\documentclass{article}\title{Demo}\begin{document}\section{Intro}We study \(a + b\) with \cite{vaswani2017}.\begin{thebibliography}{9}\bibitem{vaswani2017}A. Vaswani.\newblock Attention Is All You Need.\newblock 2017.\end{thebibliography}\end{document}"""
v2_tex = r"""\documentclass{article}\title{Demo}\begin{document}\section{Intro}We study $a+b$ with \cite{vit_transformer}.\section{Method}New method.\begin{thebibliography}{9}\bibitem{vit_transformer}A. Vaswani.\newblock Attention Is All You Need.\newblock 2017. 10.48550/arXiv.1706.03762.\end{thebibliography}\end{document}"""
h1, h2 = parse_hierarchy(v1_tex), parse_hierarchy(v2_tex)
bib_rows, mappings, conflicts = unify_entries('demo', [*parse_bibitems(v1_tex, 1), *parse_bibitems(v2_tex, 2)])
els, edges, _ = deduplicate('demo', {1: h1, 2: h2}, mappings)
print('Citation mapping sau gộp 2 version:', mappings)
display(pd.DataFrame(bib_rows).drop(columns=['arxiv_id']))
print(f'Elements trước dedup: {len(h1.nodes)+len(h2.nodes)} -> sau dedup: {len(els)} | Cạnh hierarchy 2 version: {len(edges)}')

In [ ]:
dedup = pd.DataFrame(m2['deduplication']).T
dedup['removed'] = dedup['before'] - dedup['after']
dedup['reduction_%'] = (100 * dedup['reduction_rate']).round(2)
display(Markdown('**Bảng 7.1. Hiệu quả khử trùng xuyên version trên toàn bộ 25.000 bài.**'))
display(dedup[['before', 'after', 'removed', 'reduction_%']])

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4))
for ax, key, title in [(axes[0], 'elements', 'Elements trước/sau dedup (25.000 bài)'),
                       (axes[1], 'bibtex', 'BibTeX entries trước/sau dedup (25.000 bài)')]:
    vals = [m2['deduplication'][key]['before'], m2['deduplication'][key]['after']]
    bars = ax.bar(['Trước dedup', 'Sau dedup'], vals, color=['#768a9a', '#2f628e'])
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, v * 1.02, f'{v:,}', ha='center', fontsize=9)
    ax.set_ylim(0, max(vals) * 1.15); ax.set_title(title)
plt.tight_layout(); plt.show()
display(Markdown(f"Trên toàn tập 25.000 bài ghi nhận **{m2['unresolved_cites']:,} citation key chưa resolve theo version** "
                 f"và **{m2['bibliography_conflicts']:,} sự kiện xung đột trường bibliography** (giữ nguyên giá trị đầu kèm bản ghi xung đột)."))

## Tài liệu và mã nguồn tham chiếu

- [Đặc tả chính thức M2](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/parsing/).
- [arXiv API User's Manual](https://info.arxiv.org/help/api/user-manual.html) & [Terms of Use](https://info.arxiv.org/help/api/tou.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã nguồn triển khai: [src/parser/normalize.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/normalize.py).
- Mã nguồn triển khai: [src/parser/bibliography.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/bibliography.py).
- Mã nguồn triển khai: [src/parser/deduplicate.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/deduplicate.py).
- Mã nguồn triển khai: [src/parser/statistics.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/statistics.py).